# Chapter 8: 例外處理 (Exception Handling)
本章節將介紹 Python 中的錯誤捕捉與防禦性程式設計，透過 `try-except-else-finally` 架構確保程式在遭遇異常時依然能夠穩健執行不崩潰。

## 1. 什麼是例外（Exception）？
當程式執行時發生錯誤（例如除以 0、讀取不存在的檔案），預設會使程式當掉並中斷執行。透過「例外處理」，我們可以**在錯誤發生時進行攔截與客製化處理**，讓程式優雅地繼續運作或安全收尾。

## 2. 例外處理完整語法架構
Python 提供了非常嚴謹的四層過濾語法：
* **`try`**: 放置可能發生錯誤的測試程式碼。
* **`except 錯誤類型`**: 攔截特定的錯誤，並執行對應的補救措施。可依序堆疊多個。
* **`except` (或 `except Exception`)**: 攔截其餘所有未被指定到的錯誤。
* **`else`**: **當 `try` 區塊完全「沒有發生錯誤」時**才會執行的程式碼。
* **`finally`**: **無論有無發生錯誤，都「保證一定會執行」**的清除/收尾區塊。

In [ ]:
def test_division(a, b):
    print(f"\n--- 運算開始: {a} / {b} ---")
    try:
        result = a / b
    except ZeroDivisionError:
        print("【錯誤攔截】: 偵測到除數為 0 的錯誤！")
    except TypeError:
        print("【錯誤攔截】: 資料型態不合，無法執行除法！")
    else:
        print(f"【成功執行】: 計算結果為 {result}")
    finally:
        print("【保證執行】: 資源清理或報告完成。")

# 測試三種不同情境
test_division(10, 2)  # 情境 A: 正常無誤 -> 執行 try -> else -> finally
test_division(10, 0)  # 情境 B: 觸發 ZeroDivisionError -> 執行 try -> except -> finally
test_division(10, "2") # 情境 C: 觸發 TypeError -> 執行 try -> except -> finally

## 3. 多個錯誤的捕捉語法
當多個錯誤共用同一個處理邏輯時，除了逐個寫開，也可以將它們打包成一個 **元組 (Tuple)** 一起攔截。

In [ ]:
print("--- 多個錯誤打包在同一個 except 區塊 ---")
try:
    # 模擬可能同時有型態或數值問題的情境
    data = int("abc")
except (ValueError, TypeError) as e:
    print(f"捕捉到數值或型態錯誤！詳細原因: {e}")

## 4. 常見的內建例外錯誤類型 (Built-in Exceptions)
以下整理了 Python 核心常見的錯誤型態表格與對應成因：

| 錯誤型態 (Exception Type) | 經典錯誤範例 | 原因解釋 |
| :--- | :--- | :--- |
| **`ValueError`** | `int("abc")` | 傳入的參數**值**型態對但內容不符（`str`能轉`int`，但字母無法轉數字） |
| **`TypeError`** | `"5" + 5` | 執行了不支援的**資料型態**運算（字串不能直接加整數） |
| **`ZeroDivisionError`** | `10 / 0` | 任何數除以 0 均不合法 |
| **`IndexError`** | `a = [1, 2]; a[5]` | 串列或元組的**索引值**超出了安全範圍 |
| **`KeyError`** | `d = {"a": 1}; d["b"]` | 讀取字典中**不存在的鍵 (Key)** |
| **`NameError`** | `print(x)` | 使用了**未定義**或未賦值的變數名稱 |
| **`FileNotFoundError`** | `open("no_file.txt")` | 嘗試讀取在路徑中找不到的檔案 |
| **`AttributeError`** | `x = 5; x.append(3)` | 物件調用了它根本**沒有的屬性或方法**（整數沒有 append） |
| **`ImportError`** | `from math import non_exist` | 模組存在，但要匯入的函式/變數名稱不存在 |
| **`ModuleNotFoundError`** | `import fake_module` | 在環境中完全**找不到這個模組/套件** |
| **`IndentationError`** | 程式碼少縮排 | **縮排**不符合 Python 的語法規範（語法錯誤的子類） |
| **`SyntaxError`** | `if True` (漏了冒號) | 程式碼完全違反 Python **語法基本規則**，無法編譯 |

In [ ]:
# 各種錯誤類型捕捉示範
print("--- 1. IndexError 範例 ---")
try:
    lst = [1, 2]
    print(lst[5])
except IndexError as e:
    print("捕捉到 IndexError:", e)

print("\n--- 2. KeyError 範例 ---")
try:
    d = {"a": 1}
    print(d["b"])
except KeyError as e:
    print("捕捉到 KeyError，找不到鍵:", e)

print("\n--- 3. AttributeError 範例 ---")
try:
    num = 5
    num.append(3)
except AttributeError as e:
    print("捕捉到 AttributeError:", e)

## 5. 主動拋出錯誤 (`raise`) 與 自訂例外類別
除了系統自動觸發錯誤，我們也可以在商務邏輯內使用 **`raise`** 關鍵字「主動」將異常丟出去。同時，我們也能透過繼承基礎 `Exception` 類別來建立專屬的「自訂錯誤」。

In [ ]:
# A. 建立自訂例外類別 (繼承 Exception)
class InsufficientBalanceError(Exception):
    """當提款金額大於餘額時觸發的自訂錯誤"""
    pass

# B. 在邏輯函數中主動拋出錯誤 (raise)
def withdraw(balance, amount):
    if amount < 0:
        raise ValueError("金額不能為負數")
    if amount > balance:
        # 拋出我們專屬的自訂錯誤
        raise InsufficientBalanceError(f"提款失敗！您的餘額僅剩 {balance} 元，無法提領 {amount} 元")
    return balance - amount

# C. 測試捕捉自訂錯誤與拋出機制
account_balance = 5000

print("--- 情境一：傳入負數金額 ---")
try:
    withdraw(account_balance, -200)
except ValueError as e:
    print("攔截到邏輯錯誤:", e)

print("\n--- 情境二：提款金額爆表 ---")
try:
    withdraw(account_balance, 8000)
except InsufficientBalanceError as e:
    print("攔截到自訂錯誤類別:", e)